# XE4 FMHA4 forward attention

A fully worked, cell-by-cell functional port of the sycl-tla example [`examples/xe4/fmha4/xe4_fmha_fwd.cpp`](../../examples/xe4/fmha4/xe4_fmha_fwd.cpp ), runnable on the CPU.

Flash-attention forward: S = scale·Q·Kᵀ, row softmax, O = P·V — two AMMA GEMMs bridged by an LDSM warp-row softmax, fed by ADMA loads.

Every cell performs one operation on small concrete data and shows the matching Xe layout. Run them top to bottom. (A runnable script version lives in `fmha4_fwd_xe4.py`.)

## Setup — data + a tiny layout printer

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits
from tensor_layouts.atoms_xe4 import make_xe4_ldstm

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    "Print coord -> memory offset for a rank-2 layout (truncated)."
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("  ...  (%dx%d total)" % (n_rows, n_cols))

# Real FMHA4 tile is <64,128,128,128> = <Q-blk, V-dim, KV-blk, QK-dim>; we use a
# small block so the tables print. Q=queries, K=keys, V=values.
seq_q, seq_k, head = 32, 32, 16
rng = np.random.default_rng(0)
Q = rng.integers(-2, 3, size=(seq_q, head)).astype(np.float32)   # (seq_q x head)
Kk = rng.integers(-2, 3, size=(seq_k, head)).astype(np.float32)  # (seq_k x head)
V = rng.integers(-2, 3, size=(seq_k, head)).astype(np.float32)   # (seq_k x head)
scale = 1.0 / np.sqrt(head)
print("Q", Q.shape, " K", Kk.shape, " V", V.shape, " softmax scale =", round(scale, 4))

Q (32, 16)  K (32, 16)  V (32, 16)  softmax scale = 0.25


## Step 1 — ADMA load Q,K into SLM

In [2]:
# ADMA_Q / ADMA_K: load Q (and K) into SLM through the bank-swizzled core
# matrix, then read Q back to prove the mapping is loss-less.
slm = make_slm_layout_elem(sizeof_bits("bf16"), seq_q, head)
print("SLM layout:", slm, " bijective:", is_bijective(slm))
print("\n(row, head) -> SLM offset:")
show_layout(slm, seq_q, head, rl="q", cl="d")
buf = np.zeros(size(slm), dtype=np.float32)
for i in range(seq_q):
    for j in range(head):
        buf[slm(i, j)] = Q[i, j]
Q_back = np.array([[buf[slm(i, j)] for j in range(head)] for i in range(seq_q)])
print("\nSLM round-trip recovers Q:", np.array_equal(Q_back, Q))

SLM layout: ((2, 4, 4, 1), (16, 1)) : ((16, 128, 32, 512), (1, 512))  bijective: True

(row, head) -> SLM offset:
      d0   d1   d2   d3   d4   d5   d6   d7    ...
 q0   0    1    2    3    4    5    6    7     ...
 q1   16   17   18   19   20   21   22   23    ...
 q2   128  129  130  131  132  133  134  135   ...
 q3   144  145  146  147  148  149  150  151   ...
 q4   256  257  258  259  260  261  262  263   ...
 q5   272  273  274  275  276  277  278  279   ...
 q6   384  385  386  387  388  389  390  391   ...
 q7   400  401  402  403  404  405  406  407   ...
  ...  (32x16 total)

SLM round-trip recovers Q: True


## Step 2 — GEMM-1: S = scale·Q·Kᵀ (TiledMmaQK)

In [3]:
# GEMM-1  (TiledMmaQK):  S = scale * Q . K^T     (seq_q x seq_k scores)
S = scale * (Q @ Kk.T)
print("S shape", S.shape, "\n", S[:4, :8], "...")
# accumulator ownership: thread = key column, value = query row (col-major)
C = Layout((seq_k, seq_q), (seq_q, 1))
print("\nQK accumulator (thread=key, value=query) -> element offset:")
show_layout(C, seq_k, seq_q, rl="t", cl="v")

S shape (32, 32) 
 [[-1.25 -1.5   0.5   0.75  0.25 -0.75 -1.75  2.25]
 [-0.25  0.25  1.    1.    0.    0.    1.25  2.5 ]
 [-1.25  1.75  2.   -2.    0.25 -3.   -1.75 -1.5 ]
 [ 3.75 -3.5   0.25  0.   -0.25 -1.   -0.5   0.  ]] ...

QK accumulator (thread=key, value=query) -> element offset:
      v0   v1   v2   v3   v4   v5   v6   v7    ...
 t0   0    1    2    3    4    5    6    7     ...
 t1   32   33   34   35   36   37   38   39    ...
 t2   64   65   66   67   68   69   70   71    ...
 t3   96   97   98   99   100  101  102  103   ...
 t4   128  129  130  131  132  133  134  135   ...
 t5   160  161  162  163  164  165  166  167   ...
 t6   192  193  194  195  196  197  198  199   ...
 t7   224  225  226  227  228  229  230  231   ...
  ...  (32x32 total)


## Step 3 — row softmax: S → P

In [4]:
# softmax(S) over the seq_k keys of each query row (numerically stable).
row_max = S.max(axis=1, keepdims=True)
S_exp = np.exp(S - row_max)
row_sum = S_exp.sum(axis=1, keepdims=True)
P = S_exp / row_sum
print("row_max:", row_max.ravel()[:6], "...")
print("P row 0 sums to", P[0].sum(), " P[0,:6]:", P[0, :6])

row_max: [3.25 6.5  4.25 3.75 3.   3.  ] ...
P row 0 sums to 1.0  P[0,:6]: [0.   0.   0.01 0.02 0.01 0.  ]


## Step 4 — LDSM warp-row (fred.max co-residence)

In [5]:
# LDSM warp-row: the fred.max/sum need each query row co-resident in one
# 32-lane subgroup. Show the lane assignment and confirm the in-warp reduction.
ldsm = make_xe4_ldstm("LDSM", VS=8, s="bf16")
print("LDSM atom:", ldsm.name, " ThrID =", size(ldsm.thr_id), "lanes")
warp = {t: S[0, t] for t in range(min(seq_k, 32))}
print("row 0 lanes 0..7:", [warp[t] for t in range(8)])
print("in-warp max =", max(warp.values()), " == numpy:", max(warp.values()) == S[0, :min(seq_k, 32)].max())

LDSM atom: XE4_LDSM_VS8_BF16  ThrID = 32 lanes
row 0 lanes 0..7: [np.float64(-1.25), np.float64(-1.5), np.float64(0.5), np.float64(0.75), np.float64(0.25), np.float64(-0.75), np.float64(-1.75), np.float64(2.25)]
in-warp max = 3.25  == numpy: True


## Step 5 — GEMM-2: O = P·V (TiledMmaPV)

In [6]:
# GEMM-2  (TiledMmaPV):  O = P . V     (seq_q x head output)
O = P @ V
print("O shape", O.shape, "\n", O[:4, :8], "...")

O shape (32, 16) 
 [[ 0.56 -0.98  0.17 -0.4   0.07  0.82  0.13 -0.49]
 [ 1.8  -0.75  0.98 -0.94  1.74 -0.1  -0.04 -0.02]
 [ 0.96  0.62 -0.35 -0.39 -0.68  0.44  0.8   0.05]
 [ 0.96  0.68 -0.28  0.34 -0.92 -0.29 -0.44  0.57]] ...


## Step 6 — ADMA store O

In [7]:
# ADMA store: write O back through the SLM core matrix and confirm loss-less.
slm_o = make_slm_layout_elem(sizeof_bits("bf16"), O.shape[0], O.shape[1])
print("O corner (to store):\n", O[:3, :6], "...")
obuf = np.zeros(size(slm_o), dtype=O.dtype)
for i in range(O.shape[0]):
    for j in range(O.shape[1]):
        obuf[slm_o(i, j)] = O[i, j]
back = np.array([[obuf[slm_o(i, j)] for j in range(O.shape[1])] for i in range(O.shape[0])])
print("store round-trip matches O:", np.array_equal(back, O))

O corner (to store):
 [[ 0.56 -0.98  0.17 -0.4   0.07  0.82]
 [ 1.8  -0.75  0.98 -0.94  1.74 -0.1 ]
 [ 0.96  0.62 -0.35 -0.39 -0.68  0.44]] ...
store round-trip matches O: True


## Recap

ADMA load → QK GEMM → softmax → LDSM warp-row → PV GEMM → store.